#### v1
make basic analysis work

In [1]:
import json
import numpy as np
from scipy.stats import binomtest

In [2]:
file_1 = "outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_gold.json"
file_2 = "outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_random.json"

In [3]:
#func extract_output_results (file_1, file_2) -> grades_1, grades_2

def extract_output_results(file_path):
    # Note: not secured against edge cases (like if key "init_grades" is anything else than ["[Correct]"] or ["[Incorrect]"])
    with open(file_path, 'r') as f:
        data = json.load(f)
    length = data["stats"]["count_total"]
    grades = np.full(length, False)
    for i in range(length):
        grade = data[f"{i}"]["init_grades"][0]
        grades[i] = grade == "[Correct]"
    return grades

grades_1 = extract_output_results(file_1)
grades_2 = extract_output_results(file_2)

In [4]:
#func calc_test_statistics (grades_1, grades_2) -> n12, n21, n_star, z0

def calc_test_statistics (grades_1, grades_2):
    n12 = 0; n21 = 0
    for i in range(len(grades_1)) :
        n12 += grades_1[i] and not grades_2[i]
        n21 += not grades_1[i] and grades_2[i]
    n_star = n12 + n21
    z0 = (n21 - n12) / np.sqrt(n_star)
    p_value = binomtest(n21, n_star, alternative='greater').pvalue

    return n12, n21, n_star, z0, p_value

calc_test_statistics(grades_1, grades_2)

(1, 45, 46, 6.487446070815474, 6.679101716144942e-13)

#### v2
make analysis work with the whole set of dataset variants

In [5]:
import json
import numpy as np
from scipy.stats import binomtest

In [6]:
#func assemble_file_paths(...) -> file_paths
def assemble_file_paths(model_name, variant_names, prompt_type):
    file_paths = []
    for dataset_type in ["gold", "random"]:
        path_list = []
        for variant in variant_names:
            path_list.append(create_paths_str(model_name, variant, prompt_type, dataset_type))
        file_paths.append(path_list)
    return file_paths



#func create_paths_str(...) -> file_path
def create_paths_str(model_name, variant_name, prompt_type, dataset_type):
    return f"outputs/{model_name}/{variant_name}/responses_{prompt_type}_synthetic_dataset_{variant_name}_{dataset_type}.json"



model_name = "claude-3-opus-20240229"
variant_names = [
    "linda_variant_one_to",
    "linda_variant_one_because",
    "linda_variant_one_sothat",
    "linda_variant_three",
]
prompt_type = "baseline"
file_paths = assemble_file_paths(model_name, variant_names, prompt_type)
file_paths

[['outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_because/responses_baseline_synthetic_dataset_linda_variant_one_because_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_sothat/responses_baseline_synthetic_dataset_linda_variant_one_sothat_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_three/responses_baseline_synthetic_dataset_linda_variant_three_gold.json'],
 ['outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_because/responses_baseline_synthetic_dataset_linda_variant_one_because_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_sothat/responses_baseline_synthetic_dataset_linda_variant_one_sothat_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_three/responses_baseline_synthetic_d

In [7]:
#func collect_grades (file_paths) -> grades
#   file_paths.type = list; file_paths.shape = (2, a); a = number of dataset variants
#   grades.type = np.array; grades.shape = (2, b); b = a * n; n = length of each grades list
def collect_grades(file_paths):
    grades = []
    for i in range(2):
        grades_list = []
        for j in range(len(file_paths[0])):
            grades_list.extend(extract_output_results(file_paths[i][j]))
        grades.append(grades_list)
    return np.array(grades)



#func extract_output_results (file_path) -> grades
def extract_output_results(file_path):
    # Note: not secured against edge cases (like if key "init_grades" is anything else than ["[Correct]"] or ["[Incorrect]"])
    with open(file_path, 'r') as f:
        data = json.load(f)
    length = data["stats"]["count_total"]
    grades = np.full(length, False)
    for i in range(length):
        grade = data[f"{i}"]["init_grades"][0]
        grades[i] = grade == "[Correct]"
    return grades



grades = collect_grades(file_paths)
grades.shape

(2, 400)

In [8]:
#func calc_test_statistics (grades) -> n12, n21, n_star, z0
#   grades.type = np.array; grades.shape = (2, b)

def calc_test_statistics (grades):
    grades_gold, grades_random = grades
    n12 = 0; n21 = 0
    for i in range(len(grades_gold)) :
        n12 += grades_gold[i] and not grades_random[i]
        n21 += not grades_gold[i] and grades_random[i]
    n_star = n12 + n21
    z0 = (n21 - n12) / np.sqrt(n_star)
    p_value = binomtest(n21, n_star, alternative='greater').pvalue
    return n12, n21, n_star, z0, p_value



calc_test_statistics(grades)

(16, 175, 191, 11.504836224149503, 2.7525638867682486e-35)

#### v3
make the analysis work with all models and prompting methods

In [9]:
import json
import numpy as np
from scipy.stats import binomtest

In [48]:
#func run_analysis (models, prompting_types, datasets) -> results_table
#   models.type = list of strings: same with prompting_types and datasets
#   results_table = [[models], [prompting_types], [n12], [n21], [n_star], [z0], [p_value], [rejections]]

def run_analysis(models, prompting_types, datasets):
    results_table = [[], [], [], [], [], [], [], [], []]
    for model in models:
        for prompt_type in prompting_types:
            file_paths = assemble_file_paths(model, datasets, prompt_type)
            grades = collect_grades(file_paths)
            n12, n21, n_star, z0, p_value = calc_test_statistics(grades)

            results_table[0].append(model)
            results_table[1].append(prompt_type)
            results_table[2].append(n12)
            results_table[3].append(n21)
            results_table[4].append(n_star)
            results_table[5].append(z0)
            results_table[6].append(p_value)
            

    # rejections
    #results_table[7].extend(np.full(len(results_table[0]), False))
    results_table[7] = do_benjamini_hochberg(results_table[6], 0.05)

    return results_table



def do_benjamini_hochberg(p_values, sig_level):
    sort_order = np.argsort(p_values)
    p_sorted = np.sort(p_values)
    count = len(p_values)
    control_line = np.linspace(sig_level, 0, count, endpoint=False)[::-1]
    rejection_boundary = np.max(np.argwhere(p_sorted <= control_line).T[0])

    rejections = np.full(count, False)
    rejections[sort_order[:rejection_boundary+1]] = True

    return rejections
    


models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "meta-llama-3-70b-instruct",
    "meta-llama-3-8b-instruct",
    "llama-2-70b-chat",
    "claude-3-opus-20240229",
    "claude-3-sonnet-20240229",
    "mistral-large-latest",
]
prompting_types = [
    "baseline",
    "zs_cot",
    "os",
    "os_cot",
    "fs",
    "fs_cot",
]
datasets = [
    "linda_variant_one_to",
    "linda_variant_one_because",
    "linda_variant_one_sothat",
    "linda_variant_three",
]
results_table = run_analysis(models, prompting_types, datasets)
print(results_table[7])

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True False  True  True  True False  True False  True
  True  True  True  True  True  True  True  True  True  True False  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True]


In [28]:
np.linspace(1, 0, 20, endpoint=False)[::-1] * 0.05

array([0.0025, 0.005 , 0.0075, 0.01  , 0.0125, 0.015 , 0.0175, 0.02  ,
       0.0225, 0.025 , 0.0275, 0.03  , 0.0325, 0.035 , 0.0375, 0.04  ,
       0.0425, 0.045 , 0.0475, 0.05  ])

In [44]:
random_list = np.sort(np.random.rand(20) * 0.06)
linear_list = np.linspace(0.05, 0, 20, endpoint=False)[::-1]
print(random_list)
print(linear_list)
where_bigger = np.argwhere(random_list <= linear_list).T[0]
print(where_bigger)
np.max(where_bigger)

[1.92191868e-06 2.00198036e-04 5.72597282e-03 7.76414750e-03
 1.29623268e-02 1.64788476e-02 1.66026010e-02 1.66788778e-02
 1.94640715e-02 2.50856354e-02 2.93181632e-02 3.43794642e-02
 3.76352586e-02 3.81701196e-02 4.53178268e-02 4.56466537e-02
 5.09721570e-02 5.26985189e-02 5.59475901e-02 5.73155339e-02]
[0.0025 0.005  0.0075 0.01   0.0125 0.015  0.0175 0.02   0.0225 0.025
 0.0275 0.03   0.0325 0.035  0.0375 0.04   0.0425 0.045  0.0475 0.05  ]
[0 1 2 3 6 7 8]


8